# Fase 1 — Post-filtro: ¿cuánta máscara se deja en la salida?

Primera fase de la narrativa **nueva**. El sistema ya está elegido —
`NM_MVDR_DSM_FB` (`mode="fb"`, análisis rectangular + síntesis Hann,
`sharpen_exp=8`, `block_update=1`, `fe_update=1`), el lazo ciego con
**un solo DTLN**— y la máscara se usa **como post-filtro** sobre la salida del
beamformer.

`block_update=1` / `fe_update=1` son el **desacople temporal** que hace realizable el
lazo en hardware con baja latencia: el frame `t` se filtra con los pesos que quedaron
listos en `t-1`, de modo que el camino crítico por período de hop (8 ms a 16 kHz) se
reduce a la FFT, dos productos punto y la síntesis, y lo caro —dos `eigh` y un `solve`
de M×M por bin— pasa a ser una etapa de *pipeline*. Con `=1` los pesos se **siguen
recalculando en todos los frames**: lo único que se retiene es un frame. Las pruebas
corren en esta configuración para que midan el sistema que efectivamente se va a
implementar, no una versión de latencia mínima que no se puede construir.

Hecho eso, queda **un solo parámetro libre**:

$$Y_{\text{post}}(k,t) \;=\; Y(k,t)\,\big[\,s + (1-s)\,m(k,t)\,\big]$$

con $m$ la máscara **cruda** del DTLN (sin el realce `sharpen_exp`, que sólo pesa
las SCM). `s = smooth` interpola entre **sin post-filtro** ($s=1$, la salida del
MVDR tal cual) y **máscara completa** ($s=0$, sustracción espectral plena).

**Por qué esta fase va primero.** El barrido de `smooth` no cambia nada del
beamformer, así que se puede resolver *antes* de compararse con nadie; y la Fase 2
(comparación contra el geométrico y el mono) necesita el sistema ya cerrado para
que la comparación sea del sistema final y no de un prototipo con una perilla suelta.

**Qué se descartó** respecto de la Fase 2 original:

- **DTLN post completo** (la cascada BF → DTLN): fuera. `apply_dtln_post=False`.
- **Filtro BAN**: fuera. `ban=False` (default), no se barre.
- Sólo queda el barrido de `smooth`. Las condiciones acústicas son **las mismas**
  que en la Fase 2 original (RT60 × iSIR × locutor × layout de interferentes).

**La decisión a tomar** es doble:

1. **Qué valor de `s`** (el que gana el trade-off Δ PESQ ↔ Δ STOI).
2. **Si se deja fijo o se programa con el SNR.** El barrido cubre RT60 e iSIR, así
   que se puede medir cuánto se pierde con un valor único frente al óptimo por celda.

**Una propiedad del sistema que hace legítima la pregunta 2:** `smooth` **no entra
en el lazo**. En `blind_feedback_stft` la máscara que realimenta el estimador de RTF
(`m_s_fb`, `m_n_fb`) y la que pesa las SCM del núcleo se calculan *antes* de aplicar
`G = s + (1-s)m`, y los pesos `W` que se devuelven tampoco la ven. O sea: el lazo
corre **idéntico** para todo `s`, y `s` es una ganancia real aplicada al final. Dos
consecuencias prácticas:

- el barrido compara exactamente el mismo filtro espacial en todas sus filas —
  ninguna diferencia es atribuible a que el lazo convergió distinto;
- moverla en línea **no puede desestabilizar nada**, lo que hace viable una
  programación *lenta* con el iSIR estimado a ciegas
  (`beamforming/MWF/wiener_postfilter.estimate_isir_db`, que sólo necesita la mezcla
  y la máscara que el sistema ya calcula).

*Salida:* `smooth_decision.json` con el valor elegido (y, si hace falta, el
schedule), que la Fase 2 levanta tal cual.

**Correr en orden:** Setup → Config → Control → Run → Preview → Análisis → Decisión → Figuras.

## Setup — ejecutar una vez por sesión de Colab
Montar Drive, clonar el repo, instalar dependencias y actualizar el código.

In [ ]:
# Import the drive module from Google Colab
from google.colab import drive

# Mount Google Drive to the virtual machine
drive.mount('/content/drive')

In [ ]:
# 3. Descargar tu código temporalmente
%cd /content
!git clone https://github.com/MatiasVereert/Vision-Aided-Beamformer.git

In [ ]:
import os, importlib, importlib.util

WHL = "/content/drive/MyDrive/colab_wheels"   # cache persistente de wheels en Drive
ip = get_ipython()

# (modulo que se importa, spec para instalar). git+ para las libs de GitHub;
# el resto por nombre. Mismos specs de siempre -> NO fuerza rebuild del cache.
PKGS = [
    ("noisereduce",     "noisereduce"),
    ("mir_eval",        "mir_eval"),
    ("pystoi",          "pystoi"),
    ("pesq",            "pesq"),
    ("paderbox",        "paderbox"),
    ("ai_edge_litert",  "ai_edge_litert"),
    ("pb_bss",          "git+https://github.com/fgnt/pb_bss.git"),
    ("pyroomacoustics", "git+https://github.com/LCAV/pyroomacoustics.git"),
    ("nara_wpe",        "git+https://github.com/fgnt/nara_wpe.git"),
    ("fast_bss_eval",   "git+https://github.com/fakufaku/fast_bss_eval.git"),
]
def _name(spec):  # nombre instalable desde el cache (sin git+/.git)
    return spec.rsplit("/", 1)[-1].replace(".git", "") if spec.startswith("git+") else spec
BUILD = [spec for _, spec in PKGS]

# 1) (Re)construir el cache de wheels en Drive SOLO si cambio la lista (manifest).
manifest = os.path.join(WHL, ".manifest.txt")
key = "\n".join(sorted(BUILD))
if (not os.path.isfile(manifest)) or open(manifest).read() != key:
    os.makedirs(WHL, exist_ok=True)
    print("[*] (Re)construyendo cache de wheels en Drive (una vez por cambio de lista)...")
    ip.system(f"pip wheel --wheel-dir={WHL} " + " ".join(BUILD))
    with open(manifest, "w") as fh:
        fh.write(key)

# 2) Instalar desde el cache PAQUETE POR PAQUETE. Si a uno le falta la wheel
#    (p.ej. Colab cambio de version de Python), NO bloquea a los demas.
for mod, spec in PKGS:
    if importlib.util.find_spec(mod) is None:
        ip.system(f"pip install --no-index --find-links={WHL} {_name(spec)}")

# 3) AUTOCURA: lo que SIGA sin poder importarse se instala desde el indice
#    (PyPI/git) y se agrega al cache para la proxima sesion.
importlib.invalidate_caches()
missing = [(m, s) for m, s in PKGS if importlib.util.find_spec(m) is None]
if missing:
    print("[!] Faltan tras el cache:", [m for m, _ in missing], "-> instalando desde el indice...")
    ip.system("pip install " + " ".join(s for _, s in missing))
    ip.system(f"pip wheel --wheel-dir={WHL} " + " ".join(s for _, s in missing))
    importlib.invalidate_caches()

# 4) Verificacion final.
faltan = [m for m, _ in PKGS if importlib.util.find_spec(m) is None]
if faltan:
    print(f"[!] SIGUEN faltando {faltan}: reinicia el runtime "
          f"(Entorno de ejecucion > Reiniciar) y reejecuta esta celda.")
else:
    print("[*] Todas las dependencias OK.")
# Nuclear (si el cache quedo inservible tras un cambio de Python de Colab):
#   !rm -rf /content/drive/MyDrive/colab_wheels   y reejecuta -> reconstruye todo.

In [ ]:
%cd /content/Vision-Aided-Beamformer
!git pull origin main

## Config (Fase 1 — barrido de `smooth`)

In [ ]:
import sys, os, numpy as np, shutil
from datetime import datetime

repo_root = '/content/Vision-Aided-Beamformer'
src_path = os.path.join(repo_root, 'src')
for p in (repo_root, src_path):
    if p not in sys.path: sys.path.append(p)
%cd {src_path}

try:
    import tensorflow as tf
    TFLITE_AVAILABLE = True
except ImportError:
    TFLITE_AVAILABLE = False

from evaluation.full_benchmark_test_dtln_mird import run_mird_grid_search
from evaluation.bf_wrappers import NM_MVDR_DSM_FB
from propagation.mird_loader import MirdDatasetProvider

m1 = os.path.join(repo_root, "src/dnn_denoise/models/model_quant_1.tflite")
m2 = os.path.join(repo_root, "src/dnn_denoise/models/model_quant_2.tflite")

# --- DTLN mono: NO se necesita en esta fase (el baseline mono es asunto de la
# Fase 2). Con los dos interpretes en None el motor saltea el Node 4.5 y ahorra
# una pasada del modelo por celda. NM_MVDR_DSM_FB no se entera: levanta su propio
# interprete para la mascara desde base_config['dtln_model_path'].
USE_DTLN_MONO = False
interpreter_1 = interpreter_2 = None
if USE_DTLN_MONO and TFLITE_AVAILABLE and os.path.exists(m1) and os.path.exists(m2):
    interpreter_1 = tf.lite.Interpreter(model_path=m1); interpreter_1.allocate_tensors()
    interpreter_2 = tf.lite.Interpreter(model_path=m2); interpreter_2.allocate_tensors()
    print("[*] DTLN mono ON (columnas dtln_alone_*).")
else:
    print("[*] DTLN mono OFF (no hace falta en esta fase).")

input_dir = "/content/drive/MyDrive/Benchmarks_tesis/inputs"
mird_dir  = "/content/drive/MyDrive/Benchmarks_tesis/rirs"
provider = MirdDatasetProvider(root_dir=mird_dir)

# ===================== PERILLAS =====================
DURATION = 15
# UNICO eje de barrido: la suavidad del post-filtro de mascara.
#   s = 1.0 -> SIN post-filtro (salida del MVDR tal cual): la referencia.
#   s = 0.0 -> mascara completa (sustraccion espectral plena): el extremo agresivo.
# 6 puntos equiespaciados para poder ver la FORMA de la curva, no solo el maximo.
SMOOTHS = [1.0, 0.8, 0.6, 0.4, 0.2, 0.0]

# Configuracion FIJA del sistema (la version nueva). No se barre nada de esto.
SYS = dict(mode="fb", win_type='rect', synth='hann', sharpen_exp=8.0,
           # Desacople temporal del calculo de los pesos respecto del filtrado, para
           # que la implementacion en hardware tenga baja latencia: el frame t se
           # filtra con los pesos que quedaron listos en t-1, asi el camino critico
           # se reduce a la FFT, dos productos punto y la sintesis. block_update=1
           # recalcula los pesos en TODOS los frames (solo retiene uno); fe_update=1
           # hace lo mismo con el front-end de la mascara, que es la mitad barata.
           block_update=1, fe_update=1)
# ===================================================

TARGETS = [os.path.join(input_dir, f) for f in [
    "p002_emo_adoration_sentences.wav",
    "p008_emo_contentment_sentences.wav",
]]
INTERF = [os.path.join(input_dir, f) for f in [
    "techno_gated commune.wav", "hairdryer_07_SH_MKH800.wav", "drill_07_RHODE_NT1.wav",
]]
# Estres espacial FIJO (no es el foco): 1 interferente y 3 simultaneos.
INTERF_CONFIGS = [
    [(45, 1.0, 0)],
    [(45, 1.0, 0), (-30, 1.0, 1), (60, 1.0, 2)],
]

base_config = {
    'fs': 16000, 'duration': DURATION, 't_early': 0.050,
    'array_center': [3.0, 3.0, 1.2], 'mird_spacing': "3-3-3-8-3-3-3",
    'snr_db': 60.0,
    'source_path': TARGETS[0], 'interf_paths': INTERF,
    # --- WPE ELIMINADO DEL SISTEMA (use_wpe=False en toda grilla). Estos escalares
    #     solo existen porque el benchmark los exige en scene_base_config; no operan.
    'wpe_taps': 5, 'wpe_delay': 2, 'wpe_alpha': 0.9999,
    'wpe_stft_size': 512, 'wpe_stft_shift': 128,
    'stft_window': 512, 'stft_overlap': 384,
    'dtln_model_path': m1,
    'eval_references': ['early'],
}
# El wrapper lee 'dtln_sharpen_exp' de la escena si existe y PISA su propio
# sharpen_exp: se chequea que no este, para que mande SYS.
assert 'dtln_sharpen_exp' not in base_config, "dtln_sharpen_exp en la escena pisaria SYS['sharpen_exp']"
# Idem con la ventana: el wrapper pasa win_type='rect' como override, asi que
# 'stft_win_type' de la escena no aplica. Se deja sin definir a proposito.

# --- GRILLA: las MISMAS condiciones acusticas de la Fase 2 original ---
param_grid = {
    'rt60':           [0.160, 0.360, 0.610],
    'isir_db':        [-5, 0, 5, 10],     # denso: ver la tendencia por SNR
    'target_angle':   [0], 'target_dist': [1.0],
    'source_path':    TARGETS,
    'interf_configs': INTERF_CONFIGS,
    'use_wpe':        [False],
    'mismatch_gain':  [0], 'mismatch_phase': [0],
    'error_angle_deg':[0.0], 'error_distance_m':[0.0],
}

# --- PROCESADORES: el MISMO sistema, un valor de smooth cada uno --------------
# El nombre lleva el valor con 2 decimales para que el analisis lo pueda parsear.
def _stag(s):
    return f"NM-MVDR_s{s:.2f}"

processors_dict = {_stag(s): NM_MVDR_DSM_FB(smooth=(None if s >= 1.0 else s), **SYS)
                   for s in SMOOTHS}
# smooth=None y smooth=1.0 son la MISMA salida (G = 1), pero None saltea la
# multiplicacion: se usa None en el extremo s=1 para que la referencia sea
# exactamente la salida cruda del beamformer.

n_cells = (len(param_grid['rt60']) * len(TARGETS) * len(INTERF_CONFIGS)
           * len(param_grid['isir_db']))
print("="*60)
print(f"FASE 1 (post-filtro) | celdas={n_cells} x {len(processors_dict)} proc "
      f"= {n_cells*len(processors_dict)} filas")
print("sistema:", SYS)
print("smooth :", SMOOTHS)
print("="*60)

### Control — `smooth` no entra en el lazo

Chequeo directo sobre el código que se va a correr, no sobre la documentación:
se procesa la misma señal con varios `smooth` y se comparan los **pesos** `w(k,t,m)`
que devuelve el wrapper. Si son idénticos bit a bit, el filtro espacial (y por lo
tanto todo el lazo: RTF estimada, SCM, núcleo de Souden) corrió **igual** en las seis
filas del barrido, y `smooth` es exactamente lo que dice ser: una ganancia real
aplicada al final.

De paso verifica que `smooth=None` y `smooth=1.0` dan la **misma** salida — que es
por qué el extremo `s=1` del barrido se instancia con `None`.

Corre sobre 3 s de ruido: es una propiedad del **código**, no un resultado acústico.

In [ ]:
# Control barato (~30 s): ruido blanco multicanal, 3 s, un pase por valor de smooth.
CHECK_LOOP = True

if CHECK_LOOP:
    _rng = np.random.default_rng(3)
    _x = _rng.standard_normal((8, 16000 * 3)) * 0.01
    _sc = {'fs': 16000, 'stft_window': 512, 'stft_overlap': 384,
           'dtln_model_path': m1, 'ref_mic_idx': 4}
    _out = {}
    for _s in [None, 1.0, 0.4, 0.0]:
        _out[_s] = NM_MVDR_DSM_FB(smooth=_s, **SYS).process(_x, _sc)
    _w0 = _out[None][1]
    _same_w = all(np.array_equal(_w0, _out[_s][1]) for _s in [1.0, 0.4, 0.0])
    _same_y = np.abs(_out[None][0] - _out[1.0][0]).max()
    print(f"pesos identicos para todo smooth : {_same_w}")
    print(f"|y(None) - y(1.0)| max           : {_same_y:.3e}")
    print(f"smooth=0.4 SI cambia la salida   : "
          f"{np.abs(_out[None][0] - _out[0.4][0]).max() > 1e-9}")
    assert _same_w, "los pesos cambian con smooth -> el post-filtro SI entra en el lazo"
    assert _same_y == 0.0, "smooth=None y smooth=1.0 deberian ser identicos"
    print("[OK] smooth es una ganancia de salida: el lazo corre igual en todo el barrido.")
else:
    print("[i] control salteado (CHECK_LOOP=False).")

### Run — barrido de `smooth`

In [ ]:
import json
# RUN_ID con SEGUNDOS (sin colisiones); results_fases/ separa de los viejos Prueba_*.
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
temp_dir  = f"/content/results_temp/F1_{RUN_ID}"
drive_dir = f"/content/drive/MyDrive/Tesis_Beamformers/results_fases/F1_postfiltro_{RUN_ID}"
os.makedirs(temp_dir, exist_ok=True); os.makedirs(drive_dir, exist_ok=True)

df_F1 = run_mird_grid_search(
    grid_params=param_grid, dataset_provider=provider, processors=processors_dict,
    scene_base_config=base_config, output_dir=temp_dir,
    interpreter_1=interpreter_1, interpreter_2=interpreter_2,
    save_catalog=False,
    # DTLN post COMPLETO descartado del sistema -> no se corre la cascada.
    apply_dtln_post=False)

# DONE.json = marca de COMPLETITUD (solo si termino OK).
_np = len(processors_dict)
json.dump({"run_id": RUN_ID, "phase": "F1", "experiment": "postfiltro_smooth",
           "status": "COMPLETE", "rows": int(len(df_F1)), "n_processors": _np,
           "n_cells": int(len(df_F1) // max(_np, 1)),
           "system": {k: str(v) for k, v in SYS.items()},
           "smooths": SMOOTHS,
           "processors": list(processors_dict.keys()),
           "grid": {k: str(v) for k, v in param_grid.items()},
           "finished_utc": datetime.utcnow().isoformat() + "Z"},
          open(os.path.join(temp_dir, "DONE.json"), "w"), indent=2, ensure_ascii=False)
shutil.copytree(temp_dir, drive_dir, dirs_exist_ok=True)
print(f"[EXITO] Fase 1 -> {drive_dir}  ({len(df_F1)} filas, {_np} proc)")

### Preview rápido (en memoria)

In [ ]:
# Delta PESQ y Delta STOI vs smooth, una curva por iSIR. Es la figura que dice si
# el optimo se mueve con el SNR (y por lo tanto si conviene programarlo).
import re, numpy as np, matplotlib.pyplot as plt

def smooth_of(name):
    m = re.search(r"_s([0-9.]+)$", str(name))
    return float(m.group(1)) if m else np.nan

_d = df_F1.replace([np.inf, -np.inf], np.nan).copy()
_d["smooth"] = _d.processor.map(smooth_of)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (col, lbl) in zip(axes, [("Delta_tot_PESQ_early", "Δ PESQ"),
                                 ("Delta_tot_STOI_early", "Δ STOI")]):
    for isir, g in _d.groupby("isir_db"):
        c = g.groupby("smooth")[col].median()
        ax.plot(c.index.values, c.values, "-o", ms=4, label=f"iSIR={isir} dB")
    ax.set_xlabel("smooth  (1 = sin post-filtro, 0 = máscara completa)")
    ax.set_ylabel(lbl); ax.grid(alpha=0.3); ax.invert_xaxis()
axes[0].legend(fontsize=8, title="mediana por celda")
plt.suptitle("Preview Fase 1 — trade-off del post-filtro vs iSIR")
plt.tight_layout(); plt.show()

## Análisis — tablas del trade-off

Todo por **mediana**: Δ SIR y Δ SAR tienen colas legítimas enormes (incluso `inf`) en
las celdas de RT60 bajo, y cualquier media queda arrastrada por ellas
(ver la nota de agregación en `tests/`). Los `inf` se pasan a `NaN` una sola vez.

In [ ]:
import pandas as pd, numpy as np, re

# Se puede releer del CSV (para analizar una corrida vieja) o usar lo que quedo en
# memoria. Si df_F1 no existe, apunta CSV_PATH a la corrida que quieras.
try:
    df = df_F1
except NameError:
    CSV_PATH = os.path.join(drive_dir, "mird_benchmark_metrics.csv")
    df = pd.read_csv(CSV_PATH)

def smooth_of(name):
    m = re.search(r"_s([0-9.]+)$", str(name))
    return float(m.group(1)) if m else np.nan

D = df.replace([np.inf, -np.inf], np.nan).copy()
D["smooth"] = D.processor.map(smooth_of)
D = D.dropna(subset=["smooth"])

MET = [("Delta_tot_PESQ_early", "PESQ"), ("Delta_tot_STOI_early", "STOI"),
       ("Delta_tot_SDR_early", "SDR"),   ("Delta_tot_SIR_early", "SIR"),
       ("Delta_tot_SAR_early", "SAR")]
COLS = [c for c, _ in MET if c in D.columns]

print("=== Δ end-to-end (MEDIANA sobre todas las escenas) por valor de smooth ===")
print("    smooth=1.00 es la referencia SIN post-filtro.\n")
tab = D.groupby("smooth")[COLS].median().rename(columns=dict(MET)).round(3)
print(tab.to_string())

print("\n=== Δ PESQ mediano por (smooth x iSIR) — ¿se mueve el óptimo con el SNR? ===")
print(D.pivot_table(index="smooth", columns="isir_db",
                    values="Delta_tot_PESQ_early", aggfunc="median").round(3).to_string())

print("\n=== Δ STOI mediano por (smooth x iSIR) — el lado caro del trade-off ===")
print(D.pivot_table(index="smooth", columns="isir_db",
                    values="Delta_tot_STOI_early", aggfunc="median").round(4).to_string())

print("\n=== Δ PESQ mediano por (smooth x RT60) — ¿se mueve con la reverberación? ===")
print(D.pivot_table(index="smooth", columns="rt60",
                    values="Delta_tot_PESQ_early", aggfunc="median").round(3).to_string())

## Decisión — valor único vs programado por SNR

**Regla de decisión** (explícita, y es la perilla del criterio, no del sistema):

> entre los `smooth` cuyo **Δ STOI mediano** no cae más de `STOI_TOL` por debajo del
> de la referencia sin post-filtro (`s = 1`), quedarse con el de **Δ PESQ mediano
> máximo**.

Es la formalización del trade-off: el post-filtro compra PESQ y puede costar
inteligibilidad, y lo que no se acepta es *romper* STOI. `STOI_TOL` está expuesto
arriba de la celda para poder mover el criterio y ver cuánto cambia la elección.

Con esa regla se calculan tres cosas:

| |qué es|
|---|---|
| `s*` **global** | un solo valor para todo el sistema |
| `s*(RT, iSIR)` | el óptimo **por celda** — cota superior inalcanzable (RT no se observa en línea) |
| `s*(iSIR)` | el óptimo por iSIR — **el schedule implementable**, con `estimate_isir_db` |

Y el número que decide: **cuánto Δ PESQ deja sobre la mesa el valor fijo** frente al
schedule por iSIR y frente al oráculo por celda. Si la diferencia es del orden del
ruido de medición, se fija `s*` y se termina; si el schedule por iSIR recupera una
fracción apreciable, se programa (lento — recordar que `smooth` no entra en el lazo).

In [ ]:
import json
import numpy as np, pandas as pd

# ------------------------------ criterio ---------------------------------
OBJ      = "Delta_tot_PESQ_early"   # se MAXIMIZA
GUARD    = "Delta_tot_STOI_early"   # no puede caer mas de STOI_TOL bajo la referencia
STOI_TOL = 0.005                    # <<< perilla del CRITERIO (no del sistema)
REF_S    = 1.00                     # referencia = sin post-filtro
# -------------------------------------------------------------------------

def pick(sub, tol=STOI_TOL):
    """Aplica la regla sobre un subconjunto de filas y devuelve el smooth elegido."""
    med = sub.groupby("smooth")[[OBJ, GUARD]].median()
    if med.empty:
        return np.nan
    ref = med.loc[REF_S, GUARD] if REF_S in med.index else med[GUARD].max()
    ok = med[med[GUARD] >= ref - tol]
    if ok.empty:                      # ningun candidato respeta el guard -> el menos malo
        ok = med.loc[[med[GUARD].idxmax()]]
    return float(ok[OBJ].idxmax())

def value_at(sub, s):
    """Δ PESQ mediano de ese subconjunto con ese smooth (NaN si no esta)."""
    m = sub[sub.smooth == s][OBJ].median()
    return float(m) if pd.notna(m) else np.nan

S_STAR      = pick(D)                                        # un valor para todo
by_cell     = D.groupby(["rt60", "isir_db"])                  # oraculo por celda
S_CELL      = by_cell.apply(pick)
S_ISIR      = D.groupby("isir_db").apply(pick)                # schedule implementable
S_RT        = D.groupby("rt60").apply(pick)                   # control: ¿depende del RT?

print(f"smooth* GLOBAL          : {S_STAR:.2f}   (STOI_TOL={STOI_TOL})")
print(f"smooth* por iSIR        : {S_ISIR.round(2).to_dict()}")
print(f"smooth* por RT60        : {S_RT.round(2).to_dict()}")
print(f"smooth* por celda (RT,iSIR):")
print(S_CELL.unstack().round(2).to_string())

# ---- cuanto cuesta cada estrategia frente al oraculo por celda -----------
rows = []
for (rt, isir), sub in by_cell:
    v_or   = value_at(sub, S_CELL.loc[(rt, isir)])
    v_fix  = value_at(sub, S_STAR)
    v_isir = value_at(sub, S_ISIR.loc[isir])
    v_off  = value_at(sub, REF_S)
    rows.append({"rt60": rt, "isir_db": isir,
                 "s_oraculo": S_CELL.loc[(rt, isir)], "PESQ_oraculo": v_or,
                 "s_fijo": S_STAR,                    "PESQ_fijo": v_fix,
                 "s_isir": S_ISIR.loc[isir],          "PESQ_isir": v_isir,
                 "PESQ_sin_PF": v_off})
C = pd.DataFrame(rows)
C["perdida_fijo"] = C.PESQ_oraculo - C.PESQ_fijo
C["perdida_isir"] = C.PESQ_oraculo - C.PESQ_isir
C["gana_PF"]      = C.PESQ_fijo    - C.PESQ_sin_PF

print("\n=== costo de simplificar (Δ PESQ, mediana por celda) ===")
print(C.round(3).to_string(index=False))
print(f"\nperdida MEDIA del valor fijo  vs oraculo por celda : {C.perdida_fijo.mean():+.4f} PESQ"
      f"  (peor celda {C.perdida_fijo.max():+.4f})")
print(f"perdida MEDIA del schedule iSIR vs oraculo por celda: {C.perdida_isir.mean():+.4f} PESQ"
      f"  (peor celda {C.perdida_isir.max():+.4f})")
print(f"recupera el schedule iSIR sobre el fijo             : "
      f"{(C.perdida_fijo - C.perdida_isir).mean():+.4f} PESQ")
print(f"lo que compra el post-filtro sobre NO usarlo        : {C.gana_PF.mean():+.4f} PESQ")

# ---- sensibilidad de la eleccion al criterio ----------------------------
print("\n=== sensibilidad de smooth* a STOI_TOL (¿la elección es del sistema o del criterio?) ===")
for tol in [0.0, 0.0025, 0.005, 0.01, 0.02, 1.0]:
    print(f"  STOI_TOL={tol:<6} -> smooth* = {pick(D, tol):.2f}")

# ---- veredicto y persistencia ------------------------------------------
# Umbral practico: por debajo de esto la ganancia del schedule no justifica la
# complejidad (es del orden de la dispersion entre celdas).
GAIN_MIN = 0.02   # PESQ
gain = float((C.perdida_fijo - C.perdida_isir).mean())
ADAPTIVE = gain >= GAIN_MIN

decision = {
    "system": dict({"class": "NM_MVDR_DSM_FB"},
                   **{k: str(v) for k, v in SYS.items()}),
    "criterio": {"objetivo": OBJ, "guard": GUARD, "stoi_tol": STOI_TOL,
                 "ref_smooth": REF_S, "gain_min_para_adaptar": GAIN_MIN},
    "smooth_fijo": float(S_STAR),
    "smooth_por_isir": {str(k): float(v) for k, v in S_ISIR.items()},
    "smooth_por_rt60": {str(k): float(v) for k, v in S_RT.items()},
    "adaptativo_recomendado": bool(ADAPTIVE),
    "ganancia_schedule_PESQ": gain,
    "perdida_fijo_vs_oraculo_PESQ": float(C.perdida_fijo.mean()),
}
print("\n" + "="*60)
print(f"DECISION: smooth = {S_STAR:.2f} "
      f"{'con schedule por iSIR' if ADAPTIVE else 'FIJO (el schedule no se justifica)'}")
print("="*60)

_out = os.path.join(drive_dir, "smooth_decision.json") if "drive_dir" in globals() \
       else "smooth_decision.json"
json.dump(decision, open(_out, "w"), indent=2, ensure_ascii=False)
C.to_csv(os.path.splitext(_out)[0] + "_costos.csv", index=False)
print("[*] guardado en", _out)

## Figuras — curva del post-filtro, trade-off y mapa de la decisión

Cuatro figuras, en el orden en que se leen en el documento:

1. **Curva por métrica vs `smooth`**, una serie por iSIR: la forma del trade-off.
2. **Frontera Δ PESQ ↔ Δ STOI**: dónde está el codo, y si `s*` cae sobre él.
3. **Mapa del óptimo por celda (RT60 × iSIR)**: si el color es plano, un valor fijo
   alcanza; si varía sólo en el eje iSIR, el schedule ciego es suficiente.
4. **Costo de simplificar**: Δ PESQ perdido por celda con el valor fijo y con el
   schedule por iSIR, contra el oráculo por celda.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

SAVE_DIR = drive_dir if "drive_dir" in globals() else "."
MET4 = [("Delta_tot_PESQ_early", "Δ PESQ"), ("Delta_tot_STOI_early", "Δ STOI"),
        ("Delta_tot_SDR_early",  "Δ SDR [dB]"), ("Delta_tot_SIR_early", "Δ SIR [dB]")]
isirs  = sorted(D.isir_db.unique())
rts    = sorted(D.rt60.unique())
ICOL   = {v: plt.cm.viridis(i / max(1, len(isirs) - 1)) for i, v in enumerate(isirs)}
RMK    = {v: m for v, m in zip(rts, ["o", "s", "^", "D", "v"])}

# ---------- FIG 1: metrica vs smooth, una serie por iSIR ----------
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (col, lbl) in zip(axes.ravel(), MET4):
    if col not in D.columns: continue
    for isir in isirs:
        g = D[D.isir_db == isir].groupby("smooth")[col].median()
        ax.plot(g.index.values, g.values, "-o", ms=4, color=ICOL[isir], label=f"{isir} dB")
    ax.axvline(S_STAR, color="k", ls="--", lw=1.2)
    ax.set_xlabel("smooth   (← más post-filtro | menos →)"); ax.set_ylabel(lbl)
    ax.grid(alpha=0.3); ax.invert_xaxis()
axes[0][0].legend(fontsize=8, title="iSIR")
fig.suptitle(f"Fase 1 — efecto del post-filtro por métrica (mediana). Línea: s* = {S_STAR:.2f}")
fig.tight_layout(); fig.savefig(os.path.join(SAVE_DIR, "F1_smooth_curvas.png"),
                                dpi=140, bbox_inches="tight"); plt.show()

# ---------- FIG 2: frontera PESQ vs STOI ----------
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (xcol, xlab) in zip(axes, [("Delta_tot_STOI_early", "Δ STOI"),
                                   ("Delta_tot_SAR_early",  "Δ SAR [dB]")]):
    if xcol not in D.columns: continue
    for rt in rts:
        sub = D[D.rt60 == rt]
        g = sub.groupby("smooth")[[xcol, "Delta_tot_PESQ_early"]].median()
        ax.plot(g[xcol].values, g["Delta_tot_PESQ_early"].values, "-", color="gray",
                lw=0.8, alpha=0.6, zorder=1)
        for s, row in g.iterrows():
            ax.scatter(row[xcol], row["Delta_tot_PESQ_early"], s=90, marker=RMK[rt],
                       c=[plt.cm.plasma(1.0 - s)], edgecolor="k", lw=0.4, zorder=3)
    ax.set_xlabel(xlab); ax.set_ylabel("Δ PESQ"); ax.grid(alpha=0.3)
h_s = [Line2D([0], [0], marker="o", ls="", mfc=plt.cm.plasma(1.0 - s), mec="k",
              ms=9, label=f"s={s:.2f}") for s in sorted(D.smooth.unique(), reverse=True)]
h_r = [Line2D([0], [0], marker=RMK[r], ls="", mfc="w", mec="k", ms=9,
              label=f"RT60={r*1000:.0f} ms") for r in rts]
axes[0].legend(handles=h_s, fontsize=7, loc="best", title="color = smooth")
axes[1].legend(handles=h_r, fontsize=7, loc="best", title="marcador = RT60")
fig.suptitle("Fase 1 — frontera del trade-off (cada punto: un smooth, mediana por RT60)")
fig.tight_layout(); fig.savefig(os.path.join(SAVE_DIR, "F1_smooth_tradeoff.png"),
                                dpi=140, bbox_inches="tight"); plt.show()

# ---------- FIG 3: mapa del optimo por celda ----------
piv_s = S_CELL.unstack()                      # index=rt60, columns=isir_db
fig, ax = plt.subplots(figsize=(6.5, 4.2))
im = ax.imshow(piv_s.values, cmap="plasma_r", aspect="auto",
               vmin=min(D.smooth), vmax=max(D.smooth))
ax.set_xticks(range(len(piv_s.columns)), [f"{c:g}" for c in piv_s.columns])
ax.set_yticks(range(len(piv_s.index)),   [f"{r*1000:.0f}" for r in piv_s.index])
ax.set_xlabel("iSIR [dB]"); ax.set_ylabel("RT60 [ms]")
for i in range(piv_s.shape[0]):
    for j in range(piv_s.shape[1]):
        ax.text(j, i, f"{piv_s.values[i, j]:.2f}", ha="center", va="center",
                fontsize=9, color="w")
fig.colorbar(im, ax=ax, label="smooth óptimo")
ax.set_title(f"Fase 1 — smooth óptimo por celda (fijo elegido: {S_STAR:.2f})")
fig.tight_layout(); fig.savefig(os.path.join(SAVE_DIR, "F1_smooth_mapa.png"),
                                dpi=140, bbox_inches="tight"); plt.show()

# ---------- FIG 4: costo de simplificar ----------
fig, ax = plt.subplots(figsize=(10, 4.2))
lbls = [f"{r.rt60*1000:.0f}ms\n{r.isir_db:+g}dB" for r in C.itertuples()]
x = np.arange(len(C)); w = 0.38
ax.bar(x - w/2, C.perdida_fijo.values, w, color="tab:red",   label=f"valor fijo s={S_STAR:.2f}")
ax.bar(x + w/2, C.perdida_isir.values, w, color="tab:blue",  label="schedule por iSIR")
ax.set_xticks(x); ax.set_xticklabels(lbls, fontsize=7)
ax.set_ylabel("Δ PESQ perdido vs óptimo por celda"); ax.axhline(0, color="k", lw=0.8)
ax.grid(alpha=0.3, axis="y"); ax.legend(fontsize=8)
ax.set_title("Fase 1 — cuánto cuesta no adaptar (menos es mejor)")
fig.tight_layout(); fig.savefig(os.path.join(SAVE_DIR, "F1_smooth_costo.png"),
                                dpi=140, bbox_inches="tight"); plt.show()